In [ ]:
import pandas as pd

# Load the dataset
file_path = "/content/ICRISAT-District Level Data.csv"   # change path if needed
df = pd.read_csv(file_path)

# Keep only important ID columns
id_columns = ['Year', 'State Name', 'Dist Name']

# Select crop-related columns (Area or Production)
crop_columns = [col for col in df.columns if 'AREA' in col.upper() or 'PRODUCTION' in col.upper()]

# Melt (wide format → long format)
melted_df = df.melt(id_vars=id_columns,
                    value_vars=crop_columns,
                    var_name="Crop_Variable",
                    value_name="Value")

# Remove rows where value is NaN or zero
melted_df = melted_df.dropna(subset=['Value'])
melted_df = melted_df[melted_df['Value'] != 0]

# Split Crop_Variable into crop name and variable type
# Example: 'RICE AREA (1000 ha)' → Crop: 'Rice', Variable: 'Area'
melted_df['Crop'] = melted_df['Crop_Variable'].apply(lambda x: x.split(' ')[0].strip().capitalize())
melted_df['Type'] = melted_df['Crop_Variable'].apply(lambda x: 'Area' if 'AREA' in x.upper() else 'Production')

# Display sample of cleaned melted data
print("✅ Melted and cleaned data sample:")
print(melted_df.head())


✅ Melted and cleaned data sample:
   Year    State Name Dist Name        Crop_Variable   Value  Crop  Type
0  2000  Chhattisgarh      Durg  RICE AREA (1000 ha)  748.76  Rice  Area
1  2001  Chhattisgarh      Durg  RICE AREA (1000 ha)  756.26  Rice  Area
2  2002  Chhattisgarh      Durg  RICE AREA (1000 ha)  753.41  Rice  Area
3  2003  Chhattisgarh      Durg  RICE AREA (1000 ha)  766.58  Rice  Area
4  2004  Chhattisgarh      Durg  RICE AREA (1000 ha)  775.99  Rice  Area


In [ ]:
final_df = melted_df.pivot_table(
    index=['Year', 'State Name', 'Dist Name', 'Crop'],
    columns='Type',
    values='Value'
).reset_index()


In [ ]:
import pandas as pd

# ✅ Step 1: Define correct column names manually
columns = [
    'Year', 'State Name', 'Dist Name', 'Crop',
    'Area', 'Production', 'Yield',
    'Zn', 'Fe', 'Cu', 'Mn', 'B', 'S',
    'Soil_District_Key', 'Soil_Original_District_Name'
]

# ✅ Step 2: Load CSV (no header in your file)
df = pd.read_csv('/content/icrisat_soil_merged.csv', header=None, names=columns)

# ✅ Step 3: Remove invalid rows (like the one with Year = 'mum')
df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
df['Year'] = df['Year'].astype(int)

# ✅ Step 4: Clean district names (create district_key for consistency)
df['district_key'] = df['Dist Name'].astype(str).str.upper().str.strip()

# ✅ Step 5: Fix common spelling mismatches (you can add more as needed)
district_map = {
    "GURGAON": "GURUGRAM",
    "BANGALORE": "BENGALURU",
    "BARDHAMAN": "BURDWAN",
    "MYSORE": "MYSURU",
    "NASIK": "NASHIK",
    "BIJAPUR": "VIJAYAPURA",
    "CALCUTTA": "KOLKATA",
    "MADRAS": "CHENNAI"
}
df['district_key'] = df['district_key'].replace(district_map)

# ✅ Step 6: (Optional) Clean State names too
df['State Name'] = df['State Name'].astype(str).str.title().str.strip()

# ✅ Step 7: Save cleaned final dataset
output_path = '/content/icrisat_soil_cleaned.csv'
df.to_csv(output_path, index=False)

# ✅ Step 8: Show summary
print("✅ Final dataset cleaned and saved successfully!")
print("📁 File saved as:", output_path)
print("✅ New shape:", df.shape)
print("\n🔹 Sample rows:")
print(df.head())


/tmp/ipython-input-281841497.py:12: DtypeWarning: Columns (0,4,5,6,7,8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/icrisat_soil_merged.csv', header=None, names=columns)


✅ Final dataset cleaned and saved successfully!
📁 File saved as: /content/icrisat_soil_cleaned.csv
✅ New shape: (109783, 16)

🔹 Sample rows:
   Year      State Name   Dist Name           Crop   Area Production  \
1  2000  Andhra Pradesh  Ananthapur         Castor   9.32       2.83   
2  2000  Andhra Pradesh  Ananthapur       Chickpea  33.88       22.7   
3  2000  Andhra Pradesh  Ananthapur         Cotton  12.03       2.74   
4  2000  Andhra Pradesh  Ananthapur  Finger Millet   7.52      18.24   
5  2000  Andhra Pradesh  Ananthapur         Fodder   2.31        NaN   

                 Yield   Zn   Fe   Cu   Mn    B    S Soil_District_Key  \
1  0.30364806866952787  NaN  NaN  NaN  NaN  NaN  NaN        ANANTHAPUR   
2   0.6700118063754427  NaN  NaN  NaN  NaN  NaN  NaN        ANANTHAPUR   
3  0.22776392352452207  NaN  NaN  NaN  NaN  NaN  NaN        ANANTHAPUR   
4   2.4255319148936167  NaN  NaN  NaN  NaN  NaN  NaN        ANANTHAPUR   
5                  NaN  NaN  NaN  NaN  NaN  NaN  NaN    

In [ ]:
import pandas as pd

# ✅ Step 1: Load the cleaned dataset (CHANGE path if needed!)
df = pd.read_csv('/content/icrisat_soil_cleaned.csv')  # or '/mnt/data/icrisat_soil_cleaned.csv'

# ✅ Step 2: Remove rows where both Area and Production are missing
df = df[~(df['Area'].isnull() & df['Production'].isnull())]

# ✅ Step 3: Recalculate Yield if missing (Production / Area)
df['Yield'] = df.apply(
    lambda row: row['Production'] / row['Area']
    if pd.notnull(row['Production']) and pd.notnull(row['Area']) and row['Area'] != 0
    else row['Yield'], axis=1
)

# ✅ Step 4: Handle missing soil values → Fill with state-wise median
soil_columns = ['Zn', 'Fe', 'Cu', 'Mn', 'B', 'S']
for col in soil_columns:
    df[col] = df.groupby('State Name')[col].transform(lambda x: x.fillna(x.median()))

# ✅ Step 5: Save final cleaned dataset
output_path = '/content/icrisat_soil_no_missing.csv'  # Change to /mnt/data/ if required
df.to_csv(output_path, index=False)

print("✅ Missing values handled successfully!")
print("📁 Final file saved to:", output_path)
print("✅ New dataset size:", df.shape)

df.head()


✅ Missing values handled successfully!
📁 Final file saved to: /content/icrisat_soil_no_missing.csv
✅ New dataset size: (109783, 16)


,Year,State Name,Dist Name,Crop,Area,Production,Yield,Zn,Fe,Cu,Mn,B,S,Soil_District_Key,Soil_Original_District_Name,district_key
0,2000,Andhra Pradesh,Ananthapur,Castor,9.32,2.83,0.303648,78.62,78.19,98.05,91.82,88.05,94.45,ANANTHAPUR,NaN,ANANTHAPUR
1,2000,Andhra Pradesh,Ananthapur,Chickpea,33.88,22.70,0.670012,78.62,78.19,98.05,91.82,88.05,94.45,ANANTHAPUR,NaN,ANANTHAPUR
2,2000,Andhra Pradesh,Ananthapur,Cotton,12.03,2.74,0.227764,78.62,78.19,98.05,91.82,88.05,94.45,ANANTHAPUR,NaN,ANANTHAPUR
3,2000,Andhra Pradesh,Ananthapur,Finger Millet,7.52,18.24,2.425532,78.62,78.19,98.05,91.82,88.05,94.45,ANANTHAPUR,NaN,ANANTHAPUR
4,2000,Andhra Pradesh,Ananthapur,Fodder,2.31,NaN,NaN,78.62,78.19,98.05,91.82,88.05,94.45,ANANTHAPUR,NaN,ANANTHAPUR


In [ ]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv('/content/icrisat_soil_no_missing.csv')

# Define crop-season mapping
kharif_crops = ['Rice', 'Maize', 'Bajra', 'Cotton', 'Groundnut', 'Soybean', 'Jowar', 'Paddy', 'Tur']
rabi_crops = ['Wheat', 'Barley', 'Mustard', 'Gram', 'Chickpea', 'Lentil', 'Peas', 'Rapeseed']
zaid_crops = ['Watermelon', 'Muskmelon', 'Cucumber', 'Moong', 'Summer Moong', 'Vegetables']

def assign_season(crop):
    if crop in kharif_crops:
        return 'Kharif'
    elif crop in rabi_crops:
        return 'Rabi'
    elif crop in zaid_crops:
        return 'Zaid'
    else:
        return 'Unknown'  # For crops not in the dictionary

# Apply season mapping
df['Season'] = df['Crop'].apply(assign_season)

# Save updated dataset
df.to_csv('/content/icrisat_soil_season.csv', index=False)

print("✅ Season column added successfully!")
print(df[['Crop', 'Season']].head(10))


✅ Season column added successfully!
                    Crop   Season
0                 Castor  Unknown
1               Chickpea     Rabi
2                 Cotton   Kharif
3          Finger Millet  Unknown
4                 Fodder  Unknown
5                 Fruits  Unknown
6  Fruits And Vegetables  Unknown
7              Groundnut   Kharif
8         Kharif Sorghum  Unknown
9                Linseed  Unknown


In [ ]:

from catboost import CatBoostClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv('/content/icrisat_soil_season.csv')

# Define categorical features
categorical_cols = ['State Name', 'Dist Name', 'Season']

# Features & Target
features = ['Zn', 'Fe', 'Cu', 'Mn', 'B', 'S', 'Yield', 'State Name', 'Dist Name', 'Year', 'Season']
X = df[features]
y = df['Crop']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Faster CatBoost Model
model_cb = CatBoostClassifier(
    iterations=200,      # ↓ from 400 → faster
    depth=6,             # ↓ lower tree depth → faster
    learning_rate=0.1,
    loss_function='MultiClass',
    task_type='CPU',     # set to 'GPU' if GPU is available
    verbose=50
)

# Train
model_cb.fit(X_train, y_train, cat_features=[features.index(c) for c in categorical_cols])

# Predict & Evaluate
y_pred = model_cb.predict(X_test)
print("✅ Optimized CatBoost Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


//weather integeration

In [ ]:
import pandas as pd

# ✅ Step 1: Load your dataset (adjust if in Drive)
df = pd.read_csv('/content/icrisat_soil_no_missing.csv')  # change path if needed

# ✅ Step 2: Get unique State + District combinations
districts = df[['State Name', 'Dist Name']].drop_duplicates().reset_index(drop=True)

# ✅ Step 3: Add empty latitude and longitude columns
districts['Latitude'] = None
districts['Longitude'] = None

# ✅ Step 4: Save to /content folder (works in Colab)
output_path = '/content/india_districts_latlon.csv'
districts.to_csv(output_path, index=False)

print("✅ File saved at:", output_path)


In [ ]:
!pip install geopy tqdm

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

# ✅ Load your CSV
file_path = "/content/india_districts_latlon.csv"
df = pd.read_csv(file_path)

# ✅ Standardize column names
df.columns = df.columns.str.strip()
df.rename(columns={
    "State Name": "state",
    "Dist Name": "district",
    "Latitude": "lat",
    "Longitude": "lon"
}, inplace=True)

# ✅ Geocoder setup
geolocator = Nominatim(user_agent="district_locator", timeout=5)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# ✅ Function to fetch lat,lon for missing values
def get_lat_lon(row):
    if pd.notnull(row['lat']) and pd.notnull(row['lon']):
        return row['lat'], row['lon']  # already filled
    try:
        location = geocode(f"{row['district']}, {row['state']}, India")
        if location:
            return location.latitude, location.longitude
    except:
        pass
    return None, None

# ✅ Apply only where lat or lon is missing
tqdm.pandas()
df[['lat', 'lon']] = df.progress_apply(
    lambda row: pd.Series(get_lat_lon(row)), axis=1
)

# ✅ Save completed dataset
output_path = "/content/india_districts_latlon_filled.csv"
df.to_csv(output_path, index=False)

print("✅ Finished! File saved at:", output_path)


100%|██████████| 311/311 [05:12<00:00,  1.01s/it]

✅ Finished! File saved at: /content/india_districts_latlon_filled.csv


In [ ]:
# ===============================
# District-wise NASA POWER Weather (2000–2017)
# Builds seasonal features: Kharif (Jun-Sep), Rabi (Oct-Dec prev + Jan-Mar curr), Zaid (Apr-May)
# Resume-safe, incremental save.
# ===============================

!pip install -q requests pandas tqdm

import os, time, json, math
import pandas as pd
import requests
from tqdm import tqdm

# ---------- Config ----------
CENTROIDS_CSV = "/content/district_centroids_full.csv"     # must have: state,district,lat,lon
OUT_CSV       = "/content/district_weather_seasonal_2000_2017.csv"
LOG_CSV       = "/content/district_weather_fetch_log.csv"
START_YEAR    = 1999   # we fetch from 1999 so Rabi(2000) = Oct-Dec(1999) + Jan-Mar(2000)
END_YEAR      = 2017
SLEEP_SEC     = 0.25   # polite rate limit
# ----------------------------

# Load & sanitize centroids
cent = pd.read_csv(CENTROIDS_CSV)
cent.columns = cent.columns.str.strip().str.lower()
required = {"state","district","lat","lon"}
missing_cols = required - set(cent.columns)
if missing_cols:
    raise ValueError(f"Centroid file must have columns: {required}. Missing: {missing_cols}")

# Clean & bounds check (India-ish box)
cent = cent.dropna(subset=["state","district","lat","lon"]).copy()
cent["state"]    = cent["state"].astype(str).str.strip().str.upper()
cent["district"] = cent["district"].astype(str).str.strip().str.upper()
cent["lat"]      = pd.to_numeric(cent["lat"], errors="coerce")
cent["lon"]      = pd.to_numeric(cent["lon"], errors="coerce")
cent = cent.dropna(subset=["lat","lon"])
cent = cent[cent["lat"].between(6, 38) & cent["lon"].between(68, 98)].reset_index(drop=True)

print(f"✅ Valid districts to fetch: {len(cent)}")

# Prepare resume state
done_pairs = set()
if os.path.exists(OUT_CSV):
    prev = pd.read_csv(OUT_CSV)
    if not prev.empty:
        prev["key"] = prev["state"].str.upper() + "||" + prev["district"].str.upper() + "||" + prev["year"].astype(str)
        done_pairs = set(prev["key"].unique())
        print(f"🔁 Resume mode: found {len(done_pairs)} (state,district,year) rows already saved.")

# Simple fetch with retries
def fetch_power_daily(lat, lon, year, retries=3, timeout=30):
    base = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "PRECTOT,T2M",
        "latitude": f"{lat}",
        "longitude": f"{lon}",
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "JSON",
        "community": "AG",
    }
    attempt = 0
    while attempt < retries:
        try:
            r = requests.get(base, params=params, timeout=timeout)
            if r.status_code == 200:
                js = r.json()
                param = js.get("properties",{}).get("parameter",{})
                if "PRECTOT" in param and "T2M" in param and param["PRECTOT"]:
                    # Build dataframe-like dict
                    days = []
                    for date, rain in param["PRECTOT"].items():
                        t2m = param["T2M"].get(date, None)
                        days.append({"date": date, "rain": rain, "t2m": t2m})
                    return pd.DataFrame(days)
                else:
                    return None
            elif r.status_code in (429, 502, 503, 504):
                time.sleep(1.5 * (attempt+1))
            else:
                return None
        except Exception:
            time.sleep(1.5 * (attempt+1))
        attempt += 1
    return None

# Seasonal aggregator: for YEAR = Y
# Kharif(Y): Jun-Sep of Y
# Zaid(Y):   Apr-May of Y
# Rabi(Y):   Oct-Dec (Y-1) + Jan-Mar (Y)
def seasonal_from_two_years(df_prev, df_curr, Y):
    import pandas as _pd
    out = {"year": Y}

    # normalize
    for d in (df_prev, df_curr):
        if d is not None and not d.empty:
            d["date"]  = _pd.to_datetime(d["date"], format="%Y%m%d", errors="coerce")
            d["month"] = d["date"].dt.month

    # Helper
    def agg_sum_mean(d, mask):
        if d is None or d.empty:
            return 0.0, math.nan
        s = d.loc[mask, "rain"].sum() if "rain" in d else 0.0
        m = d.loc[mask, "t2m"].mean() if "t2m" in d else math.nan
        return s, m

    # Kharif(Y) from df_curr: Jun(6)-Sep(9)
    kh_sum, kh_temp = agg_sum_mean(df_curr, (df_curr["month"]>=6)&(df_curr["month"]<=9)) if df_curr is not None else (0.0, math.nan)
    out["kharif_rain"] = kh_sum
    out["kharif_temp"] = kh_temp

    # Zaid(Y) from df_curr: Apr(4)-May(5)
    zd_sum, zd_temp = agg_sum_mean(df_curr, (df_curr["month"]>=4)&(df_curr["month"]<=5)) if df_curr is not None else (0.0, math.nan)
    out["zaid_rain"] = zd_sum
    out["zaid_temp"] = zd_temp

    # Rabi(Y): Oct(10)-Dec(12) of (Y-1) + Jan(1)-Mar(3) of Y
    rabi_prev_sum, rabi_prev_temp = (0.0, math.nan)
    rabi_curr_sum, rabi_curr_temp = (0.0, math.nan)
    if df_prev is not None:
        rabi_prev_sum, rabi_prev_temp = agg_sum_mean(df_prev, (df_prev["month"]>=10)&(df_prev["month"]<=12))
    if df_curr is not None:
        rabi_curr_sum, rabi_curr_temp = agg_sum_mean(df_curr, (df_curr["month"]>=1)&(df_curr["month"]<=3))

    # Temp for Rabi: average of two parts when available
    rabi_temp_vals = [v for v in [rabi_prev_temp, rabi_curr_temp] if not math.isnan(v)]
    rabi_temp = sum(rabi_temp_vals)/len(rabi_temp_vals) if rabi_temp_vals else math.nan

    out["rabi_rain"] = (rabi_prev_sum or 0.0) + (rabi_curr_sum or 0.0)
    out["rabi_temp"] = rabi_temp

    return out

# Prepare output containers
all_rows = []
log_rows = []

# If resuming, keep already-saved so we can append only new rows
if os.path.exists(OUT_CSV):
    try:
        already = pd.read_csv(OUT_CSV)
        all_rows = already.to_dict("records")
    except Exception:
        pass

# Main loop
for _, row in tqdm(cent.iterrows(), total=len(cent)):
    state    = row["state"]
    district = row["district"]
    lat      = float(row["lat"])
    lon      = float(row["lon"])

    # Pre-fetch daily df for 1999..2017 for this district (cache dict)
    per_year = {}
    for yr in range(START_YEAR, END_YEAR+1):
        key = f"{state}||{district}||{yr}"
        # If rows for this (state,district,year) already exist, skip fetching this year
        if f"{state.upper()}||{district.upper()}||{yr}" in done_pairs:
            per_year[yr] = None  # we won't need it
            continue

        df_year = fetch_power_daily(lat, lon, yr)
        if df_year is None:
            log_rows.append({"state": state, "district": district, "year": yr, "status": "no_data_or_error"})
        per_year[yr] = df_year
        time.sleep(SLEEP_SEC)

    # Build seasonal rows only for 2000..2017 (Rabi needs prev year)
    for Y in range(2000, END_YEAR+1):
        key = f"{state.upper()}||{district.upper()}||{Y}"
        if key in done_pairs:
            continue

        prev_df = per_year.get(Y-1)
        curr_df = per_year.get(Y)
        # If current is missing entirely, we still write a row with NaNs so merge won't break
        seas = seasonal_from_two_years(prev_df, curr_df, Y)
        seas.update({"state": state, "district": district})
        all_rows.append(seas)

    # Incremental save after each district (resume-safe)
    pd.DataFrame(all_rows).to_csv(OUT_CSV, index=False)
    if log_rows:
        pd.DataFrame(log_rows).to_csv(LOG_CSV, index=False)

print(f"✅ Done. Saved seasonal weather to: {OUT_CSV}")
if os.path.exists(LOG_CSV):
    print(f"📝 Any gaps/errors logged at: {LOG_CSV}")

# Show a preview
pd.read_csv(OUT_CSV).head()


✅ Valid districts to fetch: 289
🔁 Resume mode: found 1278 (state,district,year) rows already saved.


  0%|          | 1/289 [00:02<11:38,  2.42s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# ✅ Load your soil dataset
soil_df = pd.read_csv("/content/SOIL DATA.csv")  # Change filename if needed

# ✅ 1. Rename important columns
soil_df = soil_df.rename(columns={
    "State Name": "state",
    "Dist Name": "district",
    "Year": "year"
})

# ✅ 2. Standardize casing
soil_df.columns = soil_df.columns.str.strip().str.lower().str.replace(" ", "_")

# ✅ 3. Keep only useful columns:
keep_cols = [
    "state", "district", "year",
    "nitrogen_per_ha_of_nca_(kg_per_ha)",
    "phosphate_per_ha_of_nca_(kg_per_ha)",
    "potash_per_ha_of_nca_(kg_per_ha)"
]

soil_df = soil_df[[col for col in keep_cols if col in soil_df.columns]]

# ✅ 4. Rename nutrient columns cleanly
soil_df = soil_df.rename(columns={
    "nitrogen_per_ha_of_nca_(kg_per_ha)": "n_kg_per_ha",
    "phosphate_per_ha_of_nca_(kg_per_ha)": "p_kg_per_ha",
    "potash_per_ha_of_nca_(kg_per_ha)": "k_kg_per_ha"
})

print("✅ Final Columns in Soil Dataset:", soil_df.columns)
print("✅ Number of rows:", soil_df.shape)

# ✅ 5. Show sample
display(soil_df.head())


✅ Final Columns in Soil Dataset: Index(['state', 'district', 'year', 'n_kg_per_ha', 'p_kg_per_ha',
       'k_kg_per_ha'],
      dtype='object')
✅ Number of rows: (5572, 6)


,state,district,year,n_kg_per_ha,p_kg_per_ha,k_kg_per_ha
0,Chhattisgarh,Durg,2000,34.47,17.94,5.58
1,Chhattisgarh,Durg,2001,38.25,16.98,4.52
2,Chhattisgarh,Durg,2002,32.11,16.39,5.71
3,Chhattisgarh,Durg,2003,31.39,14.16,5.29
4,Chhattisgarh,Durg,2004,47.00,22.44,8.35


In [ ]:
import pandas as pd

# 1️⃣ Load the raw soil dataset
soil = pd.read_csv('/content/SOIL DATA.csv')  # change file name if needed

# 2️⃣ Keep only useful columns
soil = soil[['State Name', 'Dist Name', 'Year',
             'NITROGEN PER HA OF NCA (Kg per ha)',
             'PHOSPHATE PER HA OF NCA (Kg per ha)',
             'POTASH PER HA OF NCA (Kg per ha)']]

# 3️⃣ Rename columns to simpler names
soil = soil.rename(columns={
    'State Name': 'state',
    'Dist Name': 'district',
    'Year': 'year',
    'NITROGEN PER HA OF NCA (Kg per ha)': 'n_kg_per_ha',
    'PHOSPHATE PER HA OF NCA (Kg per ha)': 'p_kg_per_ha',
    'POTASH PER HA OF NCA (Kg per ha)': 'k_kg_per_ha'
})

# 4️⃣ Clean names (uppercase for matching later)
soil['state'] = soil['state'].astype(str).str.upper().str.strip()
soil['district'] = soil['district'].astype(str).str.upper().str.strip()

# 5️⃣ Save cleaned soil file
soil.to_csv("/content/soil_npk_clean.csv", index=False)

print("✅ Cleaned Soil NPK File Saved!")
soil.head()


✅ Cleaned Soil NPK File Saved!


,state,district,year,n_kg_per_ha,p_kg_per_ha,k_kg_per_ha
0,CHHATTISGARH,DURG,2000,34.47,17.94,5.58
1,CHHATTISGARH,DURG,2001,38.25,16.98,4.52
2,CHHATTISGARH,DURG,2002,32.11,16.39,5.71
3,CHHATTISGARH,DURG,2003,31.39,14.16,5.29
4,CHHATTISGARH,DURG,2004,47.00,22.44,8.35


In [ ]:
# Load merged crop dataset
crop = pd.read_csv("/content/icrisat_soil_no_missing.csv")

# Rename crop dataset columns for matching
crop = crop.rename(columns={"State Name": "state", "Dist Name": "district", "Year": "year"})
crop['state'] = crop['state'].astype(str).str.upper().str.strip()
crop['district'] = crop['district'].astype(str).str.upper().str.strip()

# Load soil NPK cleaned file
soil_npk = pd.read_csv("/content/soil_npk_clean.csv")

# Merge on state, district, year
merged = crop.merge(soil_npk, on=['state', 'district', 'year'], how='left')

# Save final file
merged.to_csv("/content/final_dataset_with_soilNPK.csv", index=False)

print("✅ Final dataset created:", merged.shape)
print("✅ Missing NPK values:", merged[['n_kg_per_ha', 'p_kg_per_ha', 'k_kg_per_ha']].isna().sum())


✅ Final dataset created: (77762, 19)
✅ Missing NPK values: n_kg_per_ha    1
p_kg_per_ha    1
k_kg_per_ha    1
dtype: int64


In [ ]:
import pandas as pd

# Load your dataset
df = pd.read_csv("/content/final_dataset_with_soilNPK.csv")

# ✅ Drop unnecessary columns
drop_columns = [
    'Soil_District_Key', 'Soil_Original_District_Name', 'district_key'
]

df = df.drop(columns=[col for col in drop_columns if col in df.columns], errors='ignore')

# ✅ Check final structure
print("✅ Final Columns Used for Model:")
print(df.columns)

# Save cleaned file
df.to_csv("/content/final_dataset_filtered.csv", index=False)
print("📁 Saved as /content/final_dataset_filtered.csv")


✅ Final Columns Used for Model:
Index(['year', 'state', 'district', 'Crop', 'Area', 'Production', 'Yield',
       'Zn', 'Fe', 'Cu', 'Mn', 'B', 'S', 'n_kg_per_ha', 'p_kg_per_ha',
       'k_kg_per_ha'],
      dtype='object')
📁 Saved as /content/final_dataset_filtered.csv


In [ ]:
import pandas as pd

# Load dataset (make sure this is the merged one with NPK + Soil + Weather)
df = pd.read_csv('/content/final_dataset_filtered.csv')

# ✅ Remove unnecessary / leakage columns
columns_to_drop = [
    'Year', 'Dist Code', 'State Code', 'Soil_District_Key',
    'Soil_Original_District_Name', 'Yield'  # remove Yield unless predicting yield
]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

# ✅ Convert State, District to uppercase for consistency
df['state'] = df['state'].astype(str).str.upper().str.strip()
df['district'] = df['district'].astype(str).str.upper().str.strip()

print("✅ Data ready for training. Shape:", df.shape)


✅ Data ready for training. Shape: (77762, 15)


In [ ]:
# ✅ Input features for model
features = [
    'Zn', 'Fe', 'Cu', 'Mn', 'B', 'S',           # Soil Micronutrients
    'n_kg_per_ha', 'p_kg_per_ha', 'k_kg_per_ha', # NPK Macros
    'kharif_rain', 'kharif_temp',
    'rabi_rain', 'rabi_temp',
    'zaid_rain', 'zaid_temp',
    'state', 'district', 'Season'               # Location + Seasonal Info
]

# ✅ Remove features not available
features = [f for f in features if f in df.columns]

X = df[features]          # Features
y = df['Crop']            # Target label


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from catboost import CatBoostClassifier

# Identify categorical features
categorical_cols = ['state', 'district', 'Season']
cat_indices = [features.index(col) for col in categorical_cols if col in features]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Train CatBoost Model
model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    task_type='CPU',
    verbose=50
)

model.fit(X_train, y_train, cat_features=cat_indices)

# ✅ Evaluate
y_pred = model.predict(X_test)
print("\n🎯 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))


0:	learn: 3.3455607	total: 6.71s	remaining: 22m 15s
50:	learn: 3.1898369	total: 5m 47s	remaining: 16m 55s
100:	learn: 3.1602349	total: 11m 13s	remaining: 11m
150:	learn: 3.1336648	total: 17m 24s	remaining: 5m 39s
199:	learn: 3.1145835	total: 23m 25s	remaining: 0us

🎯 Accuracy: 0.03806339612936411

📊 Classification Report:
                        precision    recall  f1-score   support

               Barley       0.04      0.07      0.05       428
               Castor       0.04      0.02      0.03       260
             Chickpea       0.04      0.07      0.05       729
               Cotton       0.03      0.07      0.04       353
        Finger Millet       0.03      0.01      0.01       232
               Fodder       0.04      0.02      0.03       563
               Fruits       0.04      0.02      0.03       661
Fruits And Vegetables       0.04      0.08      0.05       640
            Groundnut       0.04      0.03      0.03       654
       Kharif Sorghum       0.06      0.02  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# ✅ Crop Recommendation Pipeline - Colab Ready (One Cell Version)

import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ✅ Install CatBoost if not already installed
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install catboost
    from catboost import CatBoostClassifier

# ✅ --------- SET YOUR CSV FILE PATH HERE ------------ ✅
data_path = "/content/final_dataset_with_season.csv"  # Change path if needed
# ------------------------------------------------------

def topk_accuracy(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    matches = [(yt in topk[i]) for i, yt in enumerate(y_true)]
    return np.mean(matches)

def build_class_weights(y):
    cnt = Counter(y)
    total = sum(cnt.values())
    return {c: total / (len(cnt) * n) for c, n in cnt.items()}

# ✅ Load dataset
df = pd.read_csv(data_path)
target_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"

# ✅ Define features
drop_outcome = ["Area", "Production", "Yield"]
cat_cols = [c for c in ["state", "district", "Season"] if c in df.columns]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns]
feat_cols = [c for c in (cat_cols + num_cols) if c not in drop_outcome and c != target_col]

X = df[feat_cols].copy()
y = df[target_col].astype(str)
cat_idx = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

# ✅ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ Class weights
cw = build_class_weights(y_train)
class_weights = [cw.get(cls, 1.0) for cls in y_train]

# ✅ Train CatBoost Model
model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=42,
    verbose=100,
    task_type='CPU'
)

model.fit(
    X_train, y_train,
    cat_features=cat_idx,
    sample_weight=class_weights,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# ✅ Predictions & Evaluation
y_pred = model.predict(X_test).ravel().astype(str)
y_proba = model.predict_proba(X_test)
classes = model.classes_

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
top3 = topk_accuracy(y_test.values, np.array(y_proba), np.array(classes), k=3)

print("\n=== RESULTS ===")
print(f"✅ Accuracy: {acc:.4f}")
print(f"✅ Macro F1: {macro_f1:.4f}")
print(f"✅ Top-3 Accuracy: {top3:.4f}")

# ✅ Save Model & Metadata
model.save_model("catboost_crop_recs.cbm")
with open("catboost_metadata.json", "w") as f:
    json.dump({
        "feature_columns": feat_cols,
        "categorical_columns": cat_cols,
        "target_column": target_col,
        "classes": list(classes)
    }, f, indent=2)

print("\n✅ Model Saved as: catboost_crop_recs.cbm")
print("✅ Metadata Saved as: catboost_metadata.json")
print("✅ DONE!")


0:	learn: 0.1615000	test: 0.1131435	best: 0.1131435 (0)	total: 20.1s	remaining: 1h 40m 20s
100:	learn: 0.2169086	test: 0.1315785	best: 0.1316504 (99)	total: 28m 46s	remaining: 56m 40s
200:	learn: 0.2468246	test: 0.1334767	best: 0.1337507 (185)	total: 57m 41s	remaining: 28m 25s
299:	learn: 0.2845179	test: 0.1342581	best: 0.1355838 (286)	total: 1h 28m 59s	remaining: 0us

bestTest = 0.1355837858
bestIteration = 286

Shrink model to first 287 iterations.

=== RESULTS ===
✅ Accuracy: 0.1750
✅ Macro F1: 0.1519
✅ Top-3 Accuracy: 0.4553

✅ Model Saved as: catboost_crop_recs.cbm
✅ Metadata Saved as: catboost_metadata.json
✅ DONE!


In [ ]:
# ✅ Crop Recommendation Pipeline – Now Includes Rainfall Features

import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ✅ Install CatBoost if not installed
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install catboost
    from catboost import CatBoostClassifier

# ✅ --------- SET YOUR UPDATED CSV PATH HERE ---------
data_path = "/content/final_dataset_with_rainfall_cleaned.csv"  # <<--- Use rainfall dataset
# -----------------------------------------------------

def topk_accuracy(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    return np.mean([(yt in topk[i]) for i, yt in enumerate(y_true)])

def build_class_weights(y):
    cnt = Counter(y)
    total = sum(cnt.values())
    return {c: total / (len(cnt) * n) for c, n in cnt.items()}

# ✅ Load dataset
df = pd.read_csv(data_path)
target_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"

# ✅ Drop outcome variables from features
drop_outcome = ["Area", "Production", "Yield"]

# ✅ Select categorical & numeric columns (automatically includes rainfall columns)
cat_cols = [c for c in ["state", "district", "Season"] if c in df.columns]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns]

# ✅ Final feature list (soil + rainfall + other numerics)
feat_cols = [c for c in (cat_cols + num_cols) if c not in drop_outcome and c != target_col]

X = df[feat_cols].copy()
y = df[target_col].astype(str)
cat_idx = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

# ✅ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ Class weights
cw = build_class_weights(y_train)
class_weights = [cw.get(cls, 1.0) for cls in y_train]

# ✅ Train CatBoost Model
model = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_idx,
    sample_weight=class_weights,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# ✅ Predictions & Metrics
y_pred = model.predict(X_test).ravel().astype(str)
y_proba = model.predict_proba(X_test)
classes = model.classes_

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
top3 = topk_accuracy(y_test.values, np.array(y_proba), np.array(classes), k=3)

print("\n=== RESULTS WITH RAINFALL ===")
print(f"✅ Accuracy: {acc:.4f}")
print(f"✅ Macro F1: {macro_f1:.4f}")
print(f"✅ Top-3 Accuracy: {top3:.4f}")

# ✅ Save Model & Metadata
model.save_model("catboost_crop_recs.cbm")
with open("catboost_metadata.json", "w") as f:
    json.dump({
        "feature_columns": feat_cols,
        "categorical_columns": cat_cols,
        "target_column": target_col,
        "classes": list(classes)
    }, f, indent=2)

print("\n✅ Model Saved as: catboost_crop_recs.cbm")
print("✅ Metadata Saved as: catboost_metadata.json")
print("✅ DONE!")


0:	learn: 0.1760738	test: 0.1277879	best: 0.1277879 (0)	total: 18.1s	remaining: 1h 29m 57s
100:	learn: 0.2130513	test: 0.1290772	best: 0.1292156 (99)	total: 29m 34s	remaining: 58m 17s
200:	learn: 0.2472637	test: 0.1308440	best: 0.1310582 (199)	total: 58m 45s	remaining: 28m 56s
299:	learn: 0.2900103	test: 0.1328661	best: 0.1331653 (295)	total: 1h 31m 48s	remaining: 0us

bestTest = 0.1331653356
bestIteration = 295

Shrink model to first 296 iterations.

=== RESULTS WITH RAINFALL ===
✅ Accuracy: 0.1737
✅ Macro F1: 0.1501
✅ Top-3 Accuracy: 0.4531

✅ Model Saved as: catboost_crop_recs.cbm
✅ Metadata Saved as: catboost_metadata.json
✅ DONE!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ Optimized CatBoost Model Training Script with Early Stopping

import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ✅ Install CatBoost if needed
try:
    from catboost import CatBoostClassifier
except:
    !pip install catboost
    from catboost import CatBoostClassifier

# ✅ Set Dataset Path (update if needed)
data_path = "/content/final_dataset_with_rainfall_cleaned.csv"

# ✅ Load dataset
df = pd.read_csv(data_path)
target_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"

# ✅ Feature selection
drop_outcome = ["Area", "Production", "Yield"]
cat_cols = [c for c in ["state", "district", "Season"] if c in df.columns]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns]
feat_cols = [c for c in (cat_cols + num_cols) if c not in drop_outcome and c != target_col]

X = df[feat_cols]
y = df[target_col].astype(str)

# ✅ Categorical feature indexing for CatBoost
cat_idx = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

# ✅ Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ Build Model (Optimized for 77K rows, stops early if no improvement)
model = CatBoostClassifier(
    iterations=1500,              # ⬆ Can go higher but early stopping prevents long runs
    depth=10,                     # ⬆ More complex patterns learned
    learning_rate=0.03,           # ⬇ Lower to allow more iterations
    loss_function='MultiClass',
    eval_metric='TotalF1',
    auto_class_weights='Balanced',     # ✅ Auto class imbalance handling
    random_seed=42,
    grow_policy='Lossguide',
    l2_leaf_reg=5,
    bootstrap_type='Bernoulli',
    subsample=0.7,
    verbose=100,
    early_stopping_rounds=200     # ✅ Stops if model stops improving
)

# ✅ Train Model
model.fit(
    X_train, y_train,
    cat_features=cat_idx,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# ✅ Evaluation
y_pred = model.predict(X_test).ravel()
y_proba = model.predict_proba(X_test)
classes = model.classes_

top3_acc = np.mean([y_test.iloc[i] in np.argsort(-y_proba[i])[:3] for i in range(len(y_test))])
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print("\n=== RESULTS (Optimized CatBoost) ===")
print(f"✅ Accuracy: {acc:.4f}")
print(f"✅ Macro F1: {macro_f1:.4f}")
print(f"✅ Top-3 Accuracy: {top3_acc:.4f}")

# ✅ Save Model & Metadata
model.save_model("catboost_crop_recs_optimized.cbm")
with open("catboost_metadata.json", "w") as f:
    json.dump({
        "feature_columns": feat_cols,
        "categorical_columns": cat_cols,
        "target_column": target_col,
        "classes": list(classes)
    }, f, indent=2)

print("\n✅ Model saved as: catboost_crop_recs_optimized.cbm")
print("✅ Metadata saved as: catboost_metadata.json")
print("✅ Training Done!")


0:	learn: 0.1287262	test: 0.1223211	best: 0.1223211 (0)	total: 7.97s	remaining: 3h 19m 5s
100:	learn: 0.1756824	test: 0.1644565	best: 0.1648696 (84)	total: 13m 52s	remaining: 3h 12m 16s
200:	learn: 0.1849674	test: 0.1671661	best: 0.1684537 (167)	total: 23m 12s	remaining: 2h 29m 59s
300:	learn: 0.1891681	test: 0.1691156	best: 0.1691156 (300)	total: 31m 50s	remaining: 2h 6m 50s
400:	learn: 0.1947185	test: 0.1722047	best: 0.1724619 (363)	total: 39m 8s	remaining: 1h 47m 16s
500:	learn: 0.2002985	test: 0.1730590	best: 0.1743904 (468)	total: 44m 40s	remaining: 1h 29m 5s
600:	learn: 0.2028789	test: 0.1737295	best: 0.1743904 (468)	total: 50m 14s	remaining: 1h 15m 9s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.1743903747
bestIteration = 468

Shrink model to first 469 iterations.

=== RESULTS (Optimized CatBoost) ===
✅ Accuracy: 0.1716
✅ Macro F1: 0.1464
✅ Top-3 Accuracy: 0.0000

✅ Model saved as: catboost_crop_recs_optimized.cbm
✅ Metadata saved as: catboost_metadata.js

In [ ]:
import pandas as pd

# ✅ Step 1: Load dataset freshly (important to avoid previous filtering mistakes)
df = pd.read_csv('/content/final_dataset_with_rainfall_irrigation_cleaned.csv')

print("📌 Original Shape:", df.shape)

# ✅ Step 2: Remove target leakage columns (but KEEP Yield)
cols_to_remove = ['Area', 'Production']
df = df.drop(columns=[col for col in cols_to_remove if col in df.columns])

# ✅ Step 3: Remove all irrigation columns (but DO NOT drop any rows)
irrigation_cols = [col for col in df.columns if 'irrigation_' in col.lower()]
print(f"🚿 Removing {len(irrigation_cols)} irrigation columns")
df = df.drop(columns=irrigation_cols, errors='ignore')

# ✅ (Optional) If you want to keep total irrigation as a single feature:
# df['total_irrigation'] = df[irrigation_cols].sum(axis=1)

# ✅ Step 4: Remove useless columns like 'Unnamed'
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ✅ Step 5: Verify dataset shape
print("✅ Final Shape (Rows must still be 77K+):", df.shape)
print("✅ Final Columns:", df.columns.tolist())

# ✅ Step 6: Save cleaned dataset
output = '/content/dataset_clean_no_irrigation.csv'
df.to_csv(output, index=False)
print("📁 Cleaned dataset saved at:", output)


📌 Original Shape: (77762, 43)
🚿 Removing 20 irrigation columns
✅ Final Shape (Rows must still be 77K+): (77762, 21)
✅ Final Columns: ['year', 'state', 'district', 'Crop', 'Yield', 'Zn', 'Fe', 'Cu', 'Mn', 'B', 'S', 'n_kg_per_ha', 'p_kg_per_ha', 'k_kg_per_ha', 'Crop_norm', 'Season', 'rain_kharif', 'rain_rabi', 'rain_zaid', 'Total', 'Monsoon']
📁 Cleaned dataset saved at: /content/dataset_clean_no_irrigation.csv


In [ ]:
# === Crop Prediction with Temperature + Soil + Rainfall ===
# Dataset: final_dataset_temperature_cleaned.csv (after merging temperature)

import json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Install CatBoost if needed
try:
    from catboost import CatBoostClassifier
except:
    !pip install -q catboost
    from catboost import CatBoostClassifier

# ✅ Use latest dataset (with temp + rainfall + soil)
DATA_PATH = "/content/final_dataset_temperature_cleaned.csv"

# Choose Model Mode
MODE = "top12"         # "top12" or "all28_hier"
SEED = 42

# ---------- Helpers ----------
def topk_accuracy(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    return np.mean([yt in topk[i] for i, yt in enumerate(y_true)])

def train_catboost(X_tr, y_tr, X_val, y_val, cat_idx):
    model = CatBoostClassifier(
        iterations=1600,
        depth=10,
        learning_rate=0.03,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        auto_class_weights='Balanced',
        random_seed=SEED,
        grow_policy='Lossguide',
        l2_leaf_reg=6,
        bootstrap_type='Bernoulli',
        subsample=0.75,
        verbose=200,
        early_stopping_rounds=200
    )
    model.fit(X_tr, y_tr, cat_features=cat_idx, eval_set=(X_val, y_val), use_best_model=True)
    return model

# ---------- Load Dataset ----------
df = pd.read_csv(DATA_PATH)

target_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"

# Remove leakage columns (but keep Yield)
remove_cols = ['Area', 'Production']
df = df.drop(columns=[c for c in remove_cols if c in df.columns], errors="ignore")

# ✅ Categorical & Numerical columns (Temperature included automatically)
cat_cols = [c for c in ['state', 'district', 'Season'] if c in df.columns]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns]   # Includes temp & rainfall

# Final feature set
feat_cols = [c for c in (cat_cols + num_cols) if c != target_col]

X_all = df[feat_cols]
y_all = df[target_col].astype(str)

# Categorical indices for CatBoost
cat_idx = [X_all.columns.get_loc(c) for c in cat_cols]

# ---------- 🌾 TOP-12 CROPS APPROACH ----------
if MODE == "top12":
    top12 = y_all.value_counts().head(12).index.tolist()
    mask = y_all.isin(top12)
    data = df.loc[mask].copy()
    print(f"Using TOP-12 crops → {mask.sum()} rows kept.")

    def train_season(season_name):
        d = data[data['Season'] == season_name]
        if d.empty:
            print(f"No data for season {season_name}")
            return

        X = d[feat_cols]
        y = d[target_col].astype(str)

        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=SEED
        )

        model = train_catboost(X_tr, y_tr, X_te, y_te, cat_idx)

        y_pred = model.predict(X_te).ravel().astype(str)
        y_proba = np.array(model.predict_proba(X_te))
        classes = np.array(model.classes_)

        acc = accuracy_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred, average="macro")
        top3 = topk_accuracy(y_te.values, y_proba, classes, k=3)

        print(f"\n=== {season_name.upper()} RESULTS (with Temperature + Rainfall + Soil) ===")
        print(f"Accuracy: {acc:.4f} | Macro-F1: {f1:.4f} | Top-3: {top3:.4f}")

        model.save_model(f"catboost_temp_top12_{season_name}.cbm")

    train_season("Kharif")
    train_season("Rabi")

print("\n✅ DONE with temperature-based crop prediction.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.8 MB/s eta 0:00:00
Using TOP-12 crops → 41169 rows kept.
0:	learn: 0.4267430	test: 0.4418654	best: 0.4418654 (0)	total: 234ms	remaining: 6m 13s
200:	learn: 0.5081320	test: 0.4971184	best: 0.4971184 (200)	total: 40.7s	remaining: 4m 43s
400:	learn: 0.5331774	test: 0.5109065	best: 0.5115616 (394)	total: 1m 8s	remaining: 3m 24s
600:	learn: 0.5605411	test: 0.5166489	best: 0.5166489 (600)	total: 1m 33s	remaining: 2m 35s
800:	learn: 0.5777899	test: 0.5219269	best: 0.5231298 (764)	total: 1m 57s	remaining: 1m 57s
1000:	learn: 0.5945254	test: 0.5244342	best: 0.5256358 (970)	total: 2m 22s	remaining: 1m 25s
1200:	learn: 0.6048494	test: 0.5285355	best: 0.5290604 (1144)	total: 2m 46s	remaining: 55.3s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.5290604476
bestIteration = 1144

Shrink model to first 1145 iterations.

=== KHARIF RESULTS (with Temperature + Rainfall + Soil) ===
Accuracy: 0.5271 | Macro-F1: 0.5281 | Top-3

In [ ]:
# ========================= HIERARCHICAL CROP RECOMMENDATION =========================
# Season (Kharif/Rabi/Zaid/Annual) -> Season-specific Crop model (up to all 28 crops)
# Uses CatBoost, includes NDVI/EVI lag features already present in the dataset.

import numpy as np, pandas as pd, json, warnings, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

# --- Install/Import CatBoost ---
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier

SEED = 42
DATA_PATH = "/content/final_dataset_with_ndvi_cleaned.csv"

# ------------------------------ Helpers ------------------------------
def topk_accuracy_from_proba(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    y_true_arr = np.asarray(y_true)
    return np.mean([y_true_arr[i] in topk[i] for i in range(len(y_true_arr))])

def cat_idx_from_cols(all_cols, cat_cols):
    return [all_cols.index(c) for c in cat_cols if c in all_cols]

def safe_split(X, y, test_size=0.2, random_state=SEED, stratify=True):
    """Stratified split when possible; fallback to non-stratified if class counts too small."""
    if stratify:
        try:
            return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
        except ValueError:
            pass
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=None)

def train_catboost_multiclass(X_tr, y_tr, X_va, y_va, cat_idx=None, verbose=200):
    model = CatBoostClassifier(
        iterations=200,
        depth=4,
        learning_rate=0.03,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        auto_class_weights='Balanced',
        random_seed=SEED,
        grow_policy='Lossguide',
        l2_leaf_reg=6,
        bootstrap_type='Bernoulli',
        subsample=0.75,
        verbose=verbose,
        early_stopping_rounds=200
    )
    fit_kwargs = {}
    if cat_idx is not None and len(cat_idx) > 0:
        fit_kwargs["cat_features"] = cat_idx
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, **fit_kwargs)
    return model

# ------------------------------ Load & Prepare ------------------------------
df = pd.read_csv(DATA_PATH)

# Targets / columns
crop_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"
season_col = "Season"

# Remove leakage columns for modeling
df = df.drop(columns=[c for c in ["Area", "Production"] if c in df.columns], errors="ignore")

# Sanity: keep only rows that have a known season
valid_seasons = ["Kharif", "Rabi", "Zaid", "Annual"]
df = df[df[season_col].isin(valid_seasons)].copy()

# ------------------------------ Feature sets ------------------------------
# For Season model: DO NOT use Season or Crop as features.
season_cat_cols = [c for c in ["state", "district"] if c in df.columns]
season_num_cols = df.select_dtypes(include=[np.number]).columns.tolist()  # includes NDVI/EVI lag, temp, rainfall, soil, etc.
# Exclude any potential target/leakage columns just in case
season_exclude = set([season_col, crop_col])
season_feat_cols = [c for c in (season_cat_cols + season_num_cols) if c not in season_exclude]

# For Crop models (season-specific): DO NOT use Crop; Season is constant and can be dropped.
crop_cat_cols_base = [c for c in ["state", "district"] if c in df.columns]

# ------------------------------ Season Model ------------------------------
X_season = df[season_feat_cols]
y_season = df[season_col].astype(str)

X_season_tr, X_season_te, y_season_tr, y_season_te = safe_split(X_season, y_season)

season_cat_idx = cat_idx_from_cols(list(X_season.columns), season_cat_cols)
season_model = train_catboost_multiclass(X_season_tr, y_season_tr, X_season_te, y_season_te, cat_idx=season_cat_idx)

# Season metrics
season_pred = season_model.predict(X_season_te).ravel().astype(str)
season_proba = np.array(season_model.predict_proba(X_season_te))
season_classes = np.array(season_model.classes_)

season_acc = accuracy_score(y_season_te, season_pred)
season_f1  = f1_score(y_season_te, season_pred, average="macro")
season_top3 = topk_accuracy_from_proba(y_season_te.values, season_proba, season_classes, k=3)

print("\n=== SEASON MODEL RESULTS ===")
print(f"Accuracy: {season_acc:.4f} | Macro-F1: {season_f1:.4f} | Top-3: {season_top3:.4f}")
season_model.save_model("cb_season_predictor.cbm")

# ------------------------------ Season-specific Crop Models ------------------------------
season_models = {}
season_class_lists = {}
season_metrics = {}

for s in valid_seasons:
    df_s = df[df[season_col] == s].copy()
    if df_s.empty:
        print(f"\n⚠ Skipping season '{s}' (no rows).")
        continue

    # Features for crop model
    crop_cat_cols = [c for c in crop_cat_cols_base if c in df_s.columns]
    crop_num_cols = df_s.select_dtypes(include=[np.number]).columns.tolist()
    crop_feat_cols = [c for c in (crop_cat_cols + crop_num_cols) if c != crop_col]

    Xc = df_s[crop_feat_cols]
    yc = df_s[crop_col].astype(str)

    # Some rare crops might have too few examples; safe split protects us
    Xc_tr, Xc_te, yc_tr, yc_te = safe_split(Xc, yc)

    crop_cat_idx = cat_idx_from_cols(list(Xc.columns), crop_cat_cols)
    model_s = train_catboost_multiclass(Xc_tr, yc_tr, Xc_te, yc_te, cat_idx=crop_cat_idx)
    season_models[s] = model_s
    season_class_lists[s] = np.array(model_s.classes_)

    # Metrics (within-season model)
    yc_pred = model_s.predict(Xc_te).ravel().astype(str)
    yc_proba = np.array(model_s.predict_proba(Xc_te))
    classes = np.array(model_s.classes_)

    acc = accuracy_score(yc_te, yc_pred)
    f1m = f1_score(yc_te, yc_pred, average="macro")
    top3 = topk_accuracy_from_proba(yc_te.values, yc_proba, classes, k=3)

    season_metrics[s] = {"Accuracy": acc, "MacroF1": f1m, "Top3": top3}
    print(f"\n=== {s.upper()} CROP MODEL RESULTS ===")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {f1m:.4f} | Top-3: {top3:.4f}")

    model_s.save_model(f"cb_crop_{s.lower()}.cbm")

# ------------------------------ End-to-end Hierarchical Evaluation ------------------------------
# Evaluate the full pipeline on the season test split (X_season_te)
# 1) predict season -> 2) use that season's crop model to predict crops
# Also compute an 'oracle' upper bound that uses the true season's crop model.

# Prepare a view aligned to X_season_te indices
df_test = df.loc[X_season_te.index, :].copy()
y_true_crops = df_test[crop_col].astype(str).values
true_seasons = df_test[season_col].astype(str).values

# End-to-end predictions
top1_hits = 0
top3_hits = 0
n_eval = 0

# Oracle (true-season) predictions
top1_hits_oracle = 0
top3_hits_oracle = 0
n_oracle = 0

for i, idx in enumerate(X_season_te.index):
    pred_season = season_pred[i]
    row_full = df_test.loc[idx]

    # Build features for the *predicted* season's crop model
    if pred_season in season_models:
        df_s = df[df[season_col] == pred_season]
        crop_cat_cols = [c for c in crop_cat_cols_base if c in df_s.columns]
        crop_num_cols = df_s.select_dtypes(include=[np.number]).columns.tolist()
        crop_feat_cols = [c for c in (crop_cat_cols + crop_num_cols) if c != crop_col]

        # single-row dataframe with same columns
        x_row = pd.DataFrame([row_full[crop_feat_cols].tolist()], columns=crop_feat_cols)
        model_s = season_models[pred_season]
        classes = season_class_lists[pred_season]

        proba = np.array(model_s.predict_proba(x_row))
        pred_top1 = classes[np.argmax(proba, axis=1)][0]
        top3_classes = classes[np.argsort(-proba, axis=1)[:, :3][0]]

        if y_true_crops[i] == pred_top1:
            top1_hits += 1
        if y_true_crops[i] in top3_classes:
            top3_hits += 1
        n_eval += 1

    # Oracle: use TRUE season's model (upper bound)
    true_s = true_seasons[i]
    if true_s in season_models:
        df_s2 = df[df[season_col] == true_s]
        crop_cat_cols2 = [c for c in crop_cat_cols_base if c in df_s2.columns]
        crop_num_cols2 = df_s2.select_dtypes(include=[np.number]).columns.tolist()
        crop_feat_cols2 = [c for c in (crop_cat_cols2 + crop_num_cols2) if c != crop_col]

        x_row2 = pd.DataFrame([row_full[crop_feat_cols2].tolist()], columns=crop_feat_cols2)
        model_s2 = season_models[true_s]
        classes2 = season_class_lists[true_s]

        proba2 = np.array(model_s2.predict_proba(x_row2))
        pred_top1_2 = classes2[np.argmax(proba2, axis=1)][0]
        top3_classes2 = classes2[np.argsort(-proba2, axis=1)[:, :3][0]]

        if y_true_crops[i] == pred_top1_2:
            top1_hits_oracle += 1
        if y_true_crops[i] in top3_classes2:
            top3_hits_oracle += 1
        n_oracle += 1

print("\n=== END-TO-END HIERARCHICAL METRICS (Season → Crop) ===")
if n_eval > 0:
    print(f"Top-1 Accuracy: {top1_hits/n_eval:.4f} | Top-3 Accuracy: {top3_hits/n_eval:.4f} | N={n_eval}")
else:
    print("No evaluable rows for end-to-end (check season models availability).")

print("\n=== ORACLE (True-Season) UPPER-BOUND METRICS ===")
if n_oracle > 0:
    print(f"Top-1 Accuracy: {top1_hits_oracle/n_oracle:.4f} | Top-3 Accuracy: {top3_hits_oracle/n_oracle:.4f} | N={n_oracle}")
else:
    print("No evaluable rows for oracle (check season models availability).")

print("\n✅ Saved models: cb_season_predictor.cbm, cb_crop_kharif.cbm, cb_crop_rabi.cbm, cb_crop_zaid.cbm, cb_crop_annual.cbm (only those with data).")


0:	learn: 0.8922332	test: 0.8968614	best: 0.8968614 (0)	total: 664ms	remaining: 17m 42s
200:	learn: 0.9771720	test: 0.9762909	best: 0.9763342 (199)	total: 55.6s	remaining: 6m 26s
400:	learn: 0.9870009	test: 0.9841437	best: 0.9841805 (396)	total: 1m 28s	remaining: 4m 25s
600:	learn: 0.9905635	test: 0.9883148	best: 0.9883148 (592)	total: 2m 3s	remaining: 3m 25s
800:	learn: 0.9919768	test: 0.9898939	best: 0.9898939 (791)	total: 2m 38s	remaining: 2m 38s
1000:	learn: 0.9928982	test: 0.9906028	best: 0.9906028 (1000)	total: 3m 10s	remaining: 1m 54s
1200:	learn: 0.9936042	test: 0.9910537	best: 0.9910836 (1071)	total: 3m 44s	remaining: 1m 14s
1400:	learn: 0.9940567	test: 0.9912616	best: 0.9912616 (1368)	total: 4m 15s	remaining: 36.3s
1599:	learn: 0.9944629	test: 0.9917649	best: 0.9917649 (1597)	total: 4m 47s	remaining: 0us

bestTest = 0.9917648974
bestIteration = 1597

Shrink model to first 1598 iterations.

=== SEASON MODEL RESULTS ===
Accuracy: 0.9899 | Macro-F1: 0.9876 | Top-3: 1.0000
0:	lea

In [ ]:
import pandas as pd

# --------------------------------------
# 1. LOAD YOUR ORIGINAL NDVI DATASET
# --------------------------------------
df = pd.read_csv("/content/final_dataset_with_ndvi_cleaned.csv")

# Detect crop column automatically
crop_col = "Crop_norm" if "Crop_norm" in df.columns else "Crop"


# --------------------------------------
# 2. 15-GROUP MAPPING FUNCTION
# --------------------------------------
def group_crop_15(crop):
    crop = str(crop).strip()

    # 1. Rice
    if crop == "Rice":
        return "Rice"

    # 2. Wheat
    if crop == "Wheat":
        return "Wheat"

    # 3. Maize
    if crop == "Maize":
        return "Maize"

    # 4. Other Cereals (Barley)
    if crop == "Barley":
        return "OtherCereals"

    # 5. Millets (all sorghum variants + Finger + Pearl)
    if crop in [
        "Sorghum", "Kharif Sorghum", "Rabi Sorghum",
        "Finger Millet", "Pearl Millet"
    ]:
        return "Millets"

    # 6. Chickpea
    if crop == "Chickpea":
        return "Chickpea"

    # 7. Minor Pulses
    if crop in ["Minor Pulses", "Pigeonpea"]:
        return "MinorPulses"

    # 8. Sesame
    if crop == "Sesamum":
        return "Sesame"

    # 9. Mustard
    if crop == "Rapeseed And Mustard":
        return "Mustard"

    # 10. Soybean
    if crop == "Soyabean":
        return "Soybean"

    # 11. Groundnut
    if crop == "Groundnut":
        return "Groundnut"

    # 12. Other Oilseeds (sunflower/linseed/safflower/castor/oilseeds)
    if crop in ["Sunflower", "Linseed", "Safflower", "Castor", "Oilseeds"]:
        return "OtherOilseeds"

    # 13. Vegetables
    if crop in ["Vegetables", "Onion", "Potatoes"]:
        return "Vegetables"

    # 14. Fruits
    if crop in ["Fruits", "Fruits And Vegetables"]:
        return "Fruits"

    # 15. Sugarcane (kept separate as requested)
    if crop == "Sugarcane":
        return "Sugarcane"

    return "Other"  # should never occur


# --------------------------------------
# 3. APPLY THE 15-GROUP LOGIC
# --------------------------------------
df["Crop_Group_15"] = df[crop_col].apply(group_crop_15)


# --------------------------------------
# 4. SAVE NEW DATASET WITH 15 CROP GROUPS
# --------------------------------------
output_path = "/content/final_dataset_with_ndvi_15groups.csv"
df.to_csv(output_path, index=False)

print("\n✅ Successfully created:", output_path)
print("\n=== 15 GROUP CLASS DISTRIBUTION ===")
print(df["Crop_Group_15"].value_counts())



✅ Successfully created: /content/final_dataset_with_ndvi_15groups.csv

=== 15 GROUP CLASS DISTRIBUTION ===
Crop_Group_15
Millets          10058
Vegetables        9297
OtherOilseeds     8972
MinorPulses       6985
Fruits            6740
Rice              3584
Sesame            3578
Maize             3526
Chickpea          3418
Wheat             3364
Sugarcane         3317
Groundnut         3231
Mustard           3116
OtherCereals      2102
Soybean           1826
Name: count, dtype: int64


In [ ]:
df["Crop_Group_15"].value_counts()


,count
Crop_Group_15,
Millets,10058
Vegetables,9297
OtherOilseeds,8972
MinorPulses,6985
Fruits,6740
Rice,3584
Sesame,3578
Maize,3526
Chickpea,3418


In [ ]:
# ================================================================
#           HIERARCHICAL CROP RECOMMENDATION (15 GROUPS)
# Season Prediction  →  Season-Specific 15-Group Crop Prediction
# Uses CatBoost + NDVI/EVI + Soil + Weather Features
# ================================================================

import numpy as np
import pandas as pd
import warnings, os
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ----------------------- Install CatBoost -----------------------
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier

SEED = 42
DATA_PATH = "/content/final_dataset_with_ndvi_15groups.csv"   # IMPORTANT
warnings.filterwarnings("ignore")

# --------------------------- Helpers ----------------------------
def topk_accuracy(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    y = np.asarray(y_true)
    return np.mean([y[i] in topk[i] for i in range(len(y))])

def safe_split(X, y):
    try:
        return train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    except:
        return train_test_split(X, y, test_size=0.2, random_state=SEED)

def train_cb(X_tr, y_tr, X_val, y_val, cat_idx=[], verbose=200):
    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.03,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        auto_class_weights="Balanced",
        random_seed=SEED,
        l2_leaf_reg=6,
        bootstrap_type="Bernoulli",
        subsample=0.7,
        grow_policy="Lossguide",
        verbose=verbose,
        early_stopping_rounds=100,
    )

    fit_args = {"eval_set": (X_val, y_val)}
    if len(cat_idx) > 0:
        fit_args["cat_features"] = cat_idx

    model.fit(X_tr, y_tr, use_best_model=True, **fit_args)
    return model

# ================================================================
#                     LOAD & PREPARE DATA
# ================================================================
df = pd.read_csv(DATA_PATH)

season_col = "Season"
crop_col = "Crop_Group_15"     # USE NEW 15-GROUP COLUMN

# Remove leakage columns
df = df.drop(columns=[c for c in ["Area", "Production"] if c in df.columns], errors="ignore")

valid_seasons = ["Kharif", "Rabi", "Zaid", "Annual"]
df = df[df[season_col].isin(valid_seasons)].copy()

# ================================================================
#                   SEASON PREDICTION MODEL
# ================================================================
season_cats = [c for c in ["state", "district"] if c in df.columns]
season_nums = df.select_dtypes(include=[np.number]).columns.tolist()

season_feats = [c for c in (season_cats + season_nums) if c not in [season_col, crop_col]]

X_season = df[season_feats]
y_season = df[season_col].astype(str)

X_se_tr, X_se_te, y_se_tr, y_se_te = safe_split(X_season, y_season)

cat_idx_season = [list(X_season.columns).index(c) for c in season_cats]
season_model = train_cb(X_se_tr, y_se_tr, X_se_te, y_se_te, cat_idx_season)

# ---------- Season Metrics ----------
season_pred = season_model.predict(X_se_te).ravel()
season_proba = season_model.predict_proba(X_se_te)
season_classes = np.array(season_model.classes_)

print("\n===== SEASON MODEL =====")
print("Accuracy :", round(accuracy_score(y_se_te, season_pred), 4))
print("Macro-F1 :", round(f1_score(y_se_te, season_pred, average="macro"), 4))
print("Top-3    :", round(topk_accuracy(y_se_te, season_proba, season_classes, 3), 4))

season_model.save_model("cb_season_predictor.cbm")

# ================================================================
#                 SEASON-SPECIFIC 15-GROUP CROP MODELS
# ================================================================
crop_models = {}
crop_class_map = {}

print("\n================================================")
print("           TRAINING SEASON-WISE CROP MODELS")
print("================================================")

for s in valid_seasons:
    df_s = df[df[season_col] == s]
    if df_s.empty:
        print(f"⚠ No rows for season: {s}")
        continue

    crop_cats = [c for c in ["state", "district"] if c in df_s.columns]
    crop_nums = df_s.select_dtypes(include=[np.number]).columns.tolist()

    crop_feats = [c for c in (crop_cats + crop_nums) if c != crop_col]

    Xc = df_s[crop_feats]
    yc = df_s[crop_col].astype(str)

    Xc_tr, Xc_te, yc_tr, yc_te = safe_split(Xc, yc)

    cat_idx = [list(Xc.columns).index(c) for c in crop_cats]
    model = train_cb(Xc_tr, yc_tr, Xc_te, yc_te, cat_idx)

    crop_models[s] = model
    crop_class_map[s] = np.array(model.classes_)

    # Metrics
    pred = model.predict(Xc_te).ravel()
    proba = model.predict_proba(Xc_te)

    print(f"\n===== {s.upper()} 15-GROUP CROP MODEL =====")
    print("Accuracy :", round(accuracy_score(yc_te, pred), 4))
    print("Macro-F1 :", round(f1_score(yc_te, pred, average='macro'), 4))
    print("Top-3    :", round(topk_accuracy(yc_te, proba, crop_class_map[s], 3), 4))

    model.save_model(f"cb_crop_{s.lower()}_15groups.cbm")

# ================================================================
#            END-TO-END (SEASON → CROP) EVALUATION
# ================================================================
print("\n================================================")
print("     END-TO-END (SEASON → 15-GROUP CROP) TESTING")
print("================================================")

df_test = df.loc[X_se_te.index]
true_crops = df_test[crop_col].astype(str).values
true_s = df_test[season_col].astype(str).values

top1 = top3 = 0
top1_o = top3_o = 0

for i, idx in enumerate(X_se_te.index):
    predicted_season = season_pred[i]

    row = df_test.loc[idx]

    # Predicted-season crop model
    if predicted_season in crop_models:
        feats = crop_models[predicted_season]

    df_s = df[df[season_col] == predicted_season]
    crop_cats = [c for c in ["state", "district"] if c in df_s.columns]
    crop_nums = df_s.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in (crop_cats + crop_nums) if c != crop_col]

    row_input = pd.DataFrame([row[feat_cols].tolist()], columns=feat_cols)
    model_s = crop_models[predicted_season]
    classes = crop_class_map[predicted_season]

    proba = model_s.predict_proba(row_input)
    pred1 = classes[np.argmax(proba)]
    pred_top3 = classes[np.argsort(-proba)[0][:3]]

    if true_crops[i] == pred1:
        top1 += 1
    if true_crops[i] in pred_top3:
        top3 += 1

    # Oracle (true-season)
    true_season = true_s[i]
    if true_season in crop_models:
        df_s2 = df[df[season_col] == true_season]
        crop_cats2 = [c for c in ["state", "district"] if c in df_s2.columns]
        crop_nums2 = df_s2.select_dtypes(include=[np.number]).columns.tolist()
        feat_cols2 = [c for c in (crop_cats2 + crop_nums2) if c != crop_col]

        row2 = pd.DataFrame([row[feat_cols2].tolist()], columns=feat_cols2)

        model_t = crop_models[true_season]
        classes2 = crop_class_map[true_season]
        proba2 = model_t.predict_proba(row2)

        pred1_2 = classes2[np.argmax(proba2)]
        pred3_2 = classes2[np.argsort(-proba2)[0][:3]]

        if true_crops[i] == pred1_2:
            top1_o += 1
        if true_crops[i] in pred3_2:
            top3_o += 1

N = len(X_se_te)

print("\n===== END-TO-END RESULTS =====")
print(f"Top-1 Accuracy: {top1/N:.4f}")
print(f"Top-3 Accuracy: {top3/N:.4f}")

print("\n===== ORACLE (True Season) UPPER BOUND =====")
print(f"Top-1 Accuracy: {top1_o/N:.4f}")
print(f"Top-3 Accuracy: {top3_o/N:.4f}")

print("\nSaved all models successfully.")


0:	learn: 0.8875708	test: 0.8922086	best: 0.8922086 (0)	total: 457ms	remaining: 3m 47s
200:	learn: 0.9828171	test: 0.9809990	best: 0.9810763 (198)	total: 48.4s	remaining: 1m 12s
400:	learn: 0.9911530	test: 0.9891679	best: 0.9891679 (398)	total: 1m 29s	remaining: 22s
499:	learn: 0.9925035	test: 0.9900606	best: 0.9900606 (496)	total: 1m 50s	remaining: 0us

bestTest = 0.9900605616
bestIteration = 496

Shrink model to first 497 iterations.

===== SEASON MODEL =====
Accuracy : 0.9872
Macro-F1 : 0.9844
Top-3    : 1.0

           TRAINING SEASON-WISE CROP MODELS
0:	learn: 0.2671257	test: 0.2667006	best: 0.2667006 (0)	total: 394ms	remaining: 3m 16s
200:	learn: 0.3887565	test: 0.3803963	best: 0.3804819 (196)	total: 1m 21s	remaining: 2m 1s
400:	learn: 0.4120993	test: 0.3871168	best: 0.3876768 (346)	total: 2m 36s	remaining: 38.6s
499:	learn: 0.4316790	test: 0.3958279	best: 0.3967589 (487)	total: 3m 7s	remaining: 0us

bestTest = 0.396758901
bestIteration = 487

Shrink model to first 488 iterations